## Imports & Setup

In [5]:
import os, warnings
warnings.filterwarnings('ignore')

CFG = dict(
    SEED          = 42,
    N_FOLDS       = 2,
    EPOCHS_DEB    = 2,
    BS_DEB        = 4,
    GRAD_ACCUM_DEB  = 1,
    MAX_LEN_DEB   = 512,
    LR_DEB        = 1e-5,
    MARGIN        = 0.5,
    USE_BF16      = True,
    DATA_DIR      = '/kaggle/input/competitions/smart-mcq-solver-challenge',
    OUTPUT_DIR    = '/kaggle/working',
    ARTIFACT_DIR  = '/kaggle/working/artifacts',  
    DEBERTA_MODEL = 'microsoft/deberta-v3-large'
)

os.makedirs(CFG['OUTPUT_DIR'], exist_ok=True)
os.makedirs(CFG['ARTIFACT_DIR'], exist_ok=True)
OPTS = ['A', 'B', 'C', 'D', 'E']

In [6]:
import os, sys, json, math, random, time, gc, re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast as _autocast, GradScaler as _GradScaler
from transformers import AutoTokenizer, AutoModel

from sklearn.metrics import f1_score, accuracy_score


def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_all(CFG['SEED'])

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f"  GPU: {p.name}, {p.total_memory/1e9:.1f} GB, compute {p.major}.{p.minor}")

if CFG['USE_BF16'] and DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported():
    AMP_DTYPE = torch.bfloat16
elif DEVICE.type == 'cuda':
    AMP_DTYPE = torch.float16
else:
    AMP_DTYPE = torch.float32
USE_SCALER = (AMP_DTYPE == torch.float16)
print(f"AMP dtype: {AMP_DTYPE} | GradScaler: {USE_SCALER}")

if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    torch.set_float32_matmul_precision('high')

def autocast():
    return _autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=(DEVICE.type == 'cuda'))

def make_scaler():
    return _GradScaler('cuda', enabled=USE_SCALER)

Device: cpu
AMP dtype: torch.float32 | GradScaler: False


## Load Data

In [8]:
train = pd.read_csv(f"{CFG['DATA_DIR']}/train.csv")
test  = pd.read_csv(f"{CFG['DATA_DIR']}/test.csv")
y_idx = np.array([OPTS.index(a) for a in train['answer']])

## Normalize + GroupKFold Assignment

In [7]:
_BOILERPLATE = [
    r"pick the best possible answer:?",
    r"\bcarefully\b\.?",
    r"among the listed options\.?",
    r"choose the (correct|best) (option|answer|response)\.?",
    r"select the (best|correct|most appropriate) (option|answer|response)\.?",
    r"determine the correct option:?",
    r"identify the correct statement:?",
    r"based on the given context\.?",
    r"which of the following( statements)?( is true)?",
]
_WS = re.compile(r'\s+')

def normalize_text(text):
    s = str(text)
    for pat in _BOILERPLATE:
        s = re.sub(pat, ' ', s, flags=re.IGNORECASE)
    s = _WS.sub(' ', s).strip()
    return s

for df in (train, test):
    df['prompt_n'] = df['prompt'].map(normalize_text)
    for o in OPTS:
        df[f'{o}_n'] = df[o].map(normalize_text)

print("Normalized. Sample prompt_n:")
print(" ", train['prompt_n'].iloc[0][:200])

gkf = GroupKFold(n_splits=CFG['N_FOLDS'])
train['fold'] = -1
for fold_id, (_, va_idx) in enumerate(gkf.split(train, groups=train['prompt_n'])):
    train.loc[train.index[va_idx], 'fold'] = fold_id

assert (train['fold'] >= 0).all(), 'Some rows did not get a fold assignment'
print(f"\nFold distribution:\n{train['fold'].value_counts().sort_index().to_string()}")

_dup_check = train.groupby('prompt_n')['fold'].nunique()
assert _dup_check.max() == 1, f"Leak: {(_dup_check > 1).sum()} prompts span multiple folds"
print(f"Verified: no prompt spans multiple folds.")


CONTEXT_MAP = {}


Normalized. Sample prompt_n:
  What is Martin Heidegger's view on the relationship between time and human existence?

Fold distribution:
fold
0    1000
1    1000
Verified: no prompt spans multiple folds.


## Metrics & Loss 

In [13]:

def map_at_3(scores_2d, y_idx):
    scores_2d = np.asarray(scores_2d).reshape(-1, 5)
    y_idx     = np.asarray(y_idx)
    order = (-scores_2d).argsort(axis=1)
    total = 0.0
    for i in range(len(scores_2d)):
        top3 = order[i, :3]
        if y_idx[i] in top3:
            total += 1.0 / (list(top3).index(y_idx[i]) + 1)
    return total / len(scores_2d)

def all_metrics(scores_2d, y_idx):
    scores_2d = np.asarray(scores_2d).reshape(-1, 5)
    y_idx     = np.asarray(y_idx)
    pred_top1 = scores_2d.argmax(axis=1)
    return dict(
        map3 = map_at_3(scores_2d, y_idx),
        acc  = accuracy_score(y_idx, pred_top1),
        f1   = f1_score(y_idx, pred_top1, average='macro'),
    )


def pairwise_margin_loss(scores, correct, margin=0.5):
    B = scores.size(0)
    pos = scores[torch.arange(B), correct].unsqueeze(1).expand(-1, 5)
    target = torch.ones_like(scores)
    loss = F.margin_ranking_loss(pos, scores, target, margin=margin, reduction='none')
    mask = torch.ones_like(loss); mask[torch.arange(B), correct] = 0
    return (loss * mask).sum() / mask.sum().clamp(min=1)


print("Metrics and loss defined.")

Metrics and loss defined.


## W&B Setup

In [3]:
import os
try:
    import wandb
except ImportError:
    wandb = None


USE_WANDB     = True
RUN_VERSION   = 1
RUN_NOTES     = "run_4, n_folds=2, epoch = 2"       
WANDB_PROJECT = "24f1000134-t22026"

if USE_WANDB and wandb is not None:
    try:
        from kaggle_secrets import UserSecretsClient
        wandb.login(key=UserSecretsClient().get_secret("wandb_api"))
        print("W&B connected.")
    except Exception as e:
        print(f"W&B login skipped ({e}); set Kaggle secret 'wandb_api' to enable.")
        USE_WANDB = False
elif wandb is None:
    USE_WANDB = False
    print("wandb not installed; logging disabled.")

def wandb_init(model_tag, fold):
    if not (USE_WANDB and wandb is not None):
        return None
    return wandb.init(
        project=WANDB_PROJECT,
        name=f"{model_tag}_v{RUN_VERSION}_fold{fold}",
        group=f"{model_tag}_v{RUN_VERSION}",          
        job_type="train",
        notes=RUN_NOTES,                           
        tags=[model_tag, f"v{RUN_VERSION}"],
        config={**CFG, "run_version": RUN_VERSION,
                "run_notes": RUN_NOTES, "model": model_tag, "fold": fold},
        reinit=True,
    )

def wlog(metrics, step=None):
    if USE_WANDB and wandb is not None and wandb.run is not None:
        wandb.log(metrics, step=step)

def wandb_finish():
    if USE_WANDB and wandb is not None and wandb.run is not None:
        wandb.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /teamspace/studios/this_studio/.netrc
wandb: Currently logged in as: 24f1000134 (24f1000134-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B connected.


### Uploading Model to Kagglehub

In [6]:
try:
    import kagglehub
except ImportError:
    kagglehub = None


KAGGLE_USERNAME = "kamal134134"
PUSH_TO_HUB     = True
HUB_FRAMEWORK   = "pyTorch"

HUB_HANDLES = {
    "deberta": f"{KAGGLE_USERNAME}/genai_project/{HUB_FRAMEWORK}/deberta"
}

def push_model(model_tag, folder, notes=""):

    if not PUSH_TO_HUB or kagglehub is None:
        print(f"[hub] upload skipped for '{model_tag}' (PUSH_TO_HUB=False).")
        return
    handle = HUB_HANDLES[model_tag]
    note = notes or f"v{RUN_VERSION}: {RUN_NOTES}"
    print(f"[hub] uploading '{model_tag}' -> {handle}")
    kagglehub.model_upload(handle, folder, version_notes=note)
    
    print("Done. Inference notebooks can now kagglehub.")

## HF Auth


In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF token loaded.")
else:
    print("No HF token found — public models will still download, gated ones won't.")

In [5]:
print("Downloading deberta-v3-large tokenizer...")
t = time.time()
_tok = AutoTokenizer.from_pretrained(CFG['DEBERTA_MODEL'], token=os.environ['HF_TOKEN'])
print(f"  tokenizer ({time.time()-t:.0f}s)")

print("Downloading deberta-v3-large model weights (~1.7 GB)...")
t = time.time()
_mdl = AutoModel.from_pretrained(CFG['DEBERTA_MODEL'], token=os.environ['HF_TOKEN'])
print(f"  model ({time.time()-t:.0f}s)")

del _mdl, _tok; gc.collect(); torch.cuda.empty_cache()
print("Models cached.")


  tokenizer (1s)


Loading weights: 100%|██████████| 390/390 [00:00<00:00, 28963.62it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not 

  model (1s)
Models cached.


## Model and Dataset 

In [14]:
class DebertaScorer(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        h = self.backbone.config.hidden_size
        self.head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(h, h // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(h // 2, 1),
        )

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]   
        return self.head(cls).squeeze(-1)


class DebertaDataset(Dataset):
 
    def __init__(self, df, tokenizer, max_len, context_map=None,
                 shuffle_opts=False, has_label=True, seed=0):
        self.df       = df.reset_index(drop=True)
        self.tok      = tokenizer
        self.max_len  = max_len
        self.ctx      = context_map or {}
        self.shuffle  = shuffle_opts
        self.has_label = has_label
        self.rng      = np.random.default_rng(seed)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        prompt = r['prompt_n']
        if r['id'] in self.ctx:
            prompt = self.ctx[r['id']] + ' ' + prompt
        options = [r[f'{o}_n'] for o in OPTS]
        correct = OPTS.index(r['answer']) if self.has_label else -1
        if self.shuffle:
            perm = self.rng.permutation(5)
            options = [options[i] for i in perm]
            if self.has_label:
                correct = int(np.where(perm == correct)[0][0])
        enc = self.tok([prompt]*5, options, truncation=True,
                       padding='max_length', max_length=self.max_len,
                       return_tensors='pt')
        return dict(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask'],
            correct=torch.tensor(correct, dtype=torch.long),
            id=int(r['id']),
        )


print("Model and dataset defined.")

Model and dataset defined.


## Training Function

In [10]:
def train_deberta_cv(train_df, test_df, n_folds=CFG['N_FOLDS']):


    MODEL_TAG  = 'deberta'
    artifact_dir = f"{CFG['ARTIFACT_DIR']}/{MODEL_TAG}"
    os.makedirs(artifact_dir, exist_ok=True)

    oof_scores  = np.zeros((len(train_df), 5), dtype=np.float32)
    test_scores = np.zeros((len(test_df),  5), dtype=np.float32)
    y_idx = np.array([OPTS.index(a) for a in train_df['answer']])

    tokenizer = AutoTokenizer.from_pretrained(CFG['DEBERTA_MODEL'])

    assert 'fold' in train_df.columns, "train_df must have a 'fold' column (set in Part 2)"

    for fold in range(n_folds):
        run = wandb_init(MODEL_TAG, fold)

        va_idx = np.where(train_df['fold'].values == fold)[0]
        tr_idx = np.where(train_df['fold'].values != fold)[0]
        print(f'\n[DeBERTa] === fold {fold} (train={len(tr_idx)}, val={len(va_idx)}) ===', flush=True)

        tr_ds = DebertaDataset(train_df.iloc[tr_idx], tokenizer, CFG['MAX_LEN_DEB'],
                                CONTEXT_MAP, shuffle_opts=True,  seed=fold)
        va_ds = DebertaDataset(train_df.iloc[va_idx], tokenizer, CFG['MAX_LEN_DEB'],
                                CONTEXT_MAP, shuffle_opts=False)
        te_ds = DebertaDataset(test_df, tokenizer, CFG['MAX_LEN_DEB'],
                                CONTEXT_MAP, shuffle_opts=False, has_label=False)

        tr_dl = DataLoader(tr_ds, batch_size=CFG['BS_DEB'], shuffle=True,
                           num_workers=0, pin_memory=True)
        va_dl = DataLoader(va_ds, batch_size=CFG['BS_DEB'], shuffle=False,
                           num_workers=0, pin_memory=True)
        te_dl = DataLoader(te_ds, batch_size=CFG['BS_DEB'], shuffle=False,
                           num_workers=0, pin_memory=True)

        model = DebertaScorer(CFG['DEBERTA_MODEL']).float().to(DEVICE)

        optim = AdamW(model.parameters(), lr=CFG['LR_DEB'], weight_decay=0.01)
        steps = CFG['EPOCHS_DEB'] * len(tr_dl) // CFG['GRAD_ACCUM_DEB']
        sched = CosineAnnealingLR(optim, T_max=max(steps, 1))
        accum = CFG['GRAD_ACCUM_DEB']

        best_val_map3, best_va_scores, best_te_scores = -1, None, None

        for epoch in range(CFG['EPOCHS_DEB']):
            t_epoch = time.time()
            model.train()
            train_loss, n_train = 0.0, 0
            optim.zero_grad()

            for step, batch in enumerate(tr_dl):
                B = batch['input_ids'].size(0)
                ids  = batch['input_ids'].view(B*5, -1).to(DEVICE, non_blocking=True)
                mask = batch['attention_mask'].view(B*5, -1).to(DEVICE, non_blocking=True)
                correct = batch['correct'].to(DEVICE, non_blocking=True)

                with torch.amp.autocast(device_type="cuda", dtype=torch.float32, enabled=False):
                    scores = model(ids, mask).view(B, 5)
                    loss = pairwise_margin_loss(scores, correct, margin=CFG['MARGIN']) / accum

                if torch.isfinite(loss):
                    loss.backward()
                else:
                    print(f'    [warn] step {step}: non-finite loss, skipping backward')

                if (step + 1) % accum == 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optim.step(); sched.step()
                    optim.zero_grad()

                train_loss += loss.item() * B * accum
                n_train += B

                if step % 25 == 0:
                    print(f'    step {step}/{len(tr_dl)}  '
                          f'loss={loss.item()*accum:.4f}  '
                          f'elapsed={time.time()-t_epoch:.0f}s', flush=True)

            model.eval()
            va_loss, n_val = 0.0, 0
            va_scores_all = []
            with torch.no_grad():
                for batch in va_dl:
                    B = batch['input_ids'].size(0)
                    ids  = batch['input_ids'].view(B*5, -1).to(DEVICE)
                    mask = batch['attention_mask'].view(B*5, -1).to(DEVICE)
                    correct = batch['correct'].to(DEVICE)
                    with torch.amp.autocast(device_type="cuda", dtype=torch.float32, enabled=False):
                        scores = model(ids, mask).view(B, 5)
                        loss = pairwise_margin_loss(scores, correct, margin=CFG['MARGIN'])
                    va_loss += loss.item() * B
                    n_val += B
                    va_scores_all.append(scores.cpu().float().numpy())
            va_scores_all = np.concatenate(va_scores_all)
            val_m = all_metrics(va_scores_all, y_idx[va_idx])

            tr_loss_val = train_loss / max(n_train, 1)
            va_loss_val = va_loss / max(n_val, 1)

            print(f'  ep{epoch} ({time.time()-t_epoch:.0f}s)  '
                  f'train_loss={tr_loss_val:.4f} '
                  f'val_loss={va_loss_val:.4f} val_map3={val_m["map3"]:.4f} '
                  f'val_acc={val_m["acc"]:.4f} val_f1={val_m["f1"]:.4f}', flush=True)

            wlog({
                'deb/train_loss': tr_loss_val,
                'deb/val_loss':   va_loss_val,
                'deb/val_map3':   val_m['map3'],
                'deb/val_acc':    val_m['acc'],
                'deb/val_f1':     val_m['f1'],
                'deb/fold':       fold,
                'deb/epoch':      epoch,
            })

            if val_m['map3'] > best_val_map3:
                best_val_map3  = val_m['map3']
                best_va_scores = va_scores_all
                model.eval()
                te_scores = []
                with torch.no_grad():
                    for batch in te_dl:
                        B = batch['input_ids'].size(0)
                        ids  = batch['input_ids'].view(B*5, -1).to(DEVICE)
                        mask = batch['attention_mask'].view(B*5, -1).to(DEVICE)
                        with torch.amp.autocast(device_type="cuda", dtype=torch.float32, enabled=False):
                            scores = model(ids, mask).view(B, 5)
                        te_scores.append(scores.cpu().float().numpy())
                best_te_scores = np.concatenate(te_scores)

                torch.save(model.state_dict(), f"{artifact_dir}/best_deberta_fold{fold}.pth")
                print(f"    -> new best (val_map3={best_val_map3:.4f}), checkpoint saved", flush=True)

        wlog({'deb/best_val_map3': best_val_map3})
        oof_scores[va_idx] = best_va_scores
        test_scores += best_te_scores / n_folds
        wandb_finish()
        del model; gc.collect(); torch.cuda.empty_cache()

    overall = all_metrics(oof_scores, y_idx)
    print(f'\n[DeBERTa] OOF MAP@3={overall["map3"]:.4f} '
          f'acc={overall["acc"]:.4f} f1={overall["f1"]:.4f}', flush=True)

   
    if USE_WANDB and wandb is not None:
        summary_run = wandb.init(
            project=WANDB_PROJECT,
            name=f"{MODEL_TAG}_v{RUN_VERSION}_OOF",
            group=f"{MODEL_TAG}_v{RUN_VERSION}",
            job_type="summary", tags=[MODEL_TAG, f"v{RUN_VERSION}", "oof"],
            reinit=True,
        )
        wandb.log({
            'deb/oof_map3': overall['map3'],
            'deb/oof_acc':  overall['acc'],
            'deb/oof_f1':   overall['f1'],
        })
        wandb.run.summary['oof_map3'] = overall['map3']
        wandb.run.summary['oof_acc']  = overall['acc']
        wandb.run.summary['oof_f1']   = overall['f1']
        wandb.finish()

    np.save(f"{artifact_dir}/oof_deb.npy",  oof_scores)
    np.save(f"{artifact_dir}/test_deb.npy", test_scores)
    print(f"Saved oof_deb.npy + test_deb.npy in {artifact_dir}")

    push_model(
        MODEL_TAG,
        artifact_dir,
        notes=f"v{RUN_VERSION}: {RUN_NOTES} | OOF MAP@3={overall['map3']:.4f} "
              f"acc={overall['acc']:.4f} f1={overall['f1']:.4f}"
    )

    return oof_scores, test_scores


oof_deb = test_deb = None
if CFG['RUN_DEBERTA']:
    oof_deb, test_deb = train_deberta_cv(train, test, n_folds=CFG['N_FOLDS'])
    print(f"\nTraining complete. test_deb shape={test_deb.shape}, oof_deb shape={oof_deb.shape}")

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.



[DeBERTa] === fold 0 (train=1000, val=1000) ===


Loading weights: 100%|██████████| 390/390 [00:00<00:00, 13814.53it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not 

    step 0/250  loss=0.4924  elapsed=1s
    step 25/250  loss=0.5413  elapsed=18s
    step 50/250  loss=0.5045  elapsed=34s
    step 75/250  loss=0.4359  elapsed=50s
    step 100/250  loss=0.4827  elapsed=66s
    step 125/250  loss=0.3337  elapsed=82s
    step 150/250  loss=0.1265  elapsed=98s
    step 175/250  loss=0.2845  elapsed=114s
    step 200/250  loss=0.3665  elapsed=130s
    step 225/250  loss=0.2921  elapsed=146s
  ep0 (222s)  train_loss=0.3334 val_loss=0.0829 val_map3=0.9185 val_acc=0.8600 val_f1=0.8535
    step 0/250  loss=0.1703  elapsed=1s
    step 25/250  loss=0.0456  elapsed=17s
    step 50/250  loss=0.1488  elapsed=33s
    step 75/250  loss=0.0166  elapsed=49s
    step 100/250  loss=0.0000  elapsed=65s
    step 125/250  loss=0.0000  elapsed=81s
    step 150/250  loss=0.0078  elapsed=97s
    step 175/250  loss=0.0404  elapsed=113s
    step 200/250  loss=0.0000  elapsed=129s
    step 225/250  loss=0.0000  elapsed=145s
  ep1 (222s)  train_loss=0.0347 val_loss=0.0410 val_m

deb/best_val_map3,▁
deb/epoch,▁█
deb/fold,▁▁
deb/train_loss,█▁
deb/val_acc,▁█
deb/val_f1,▁█
deb/val_loss,█▁
deb/val_map3,▁█
deb/best_val_map3,0.94467
deb/epoch,1
deb/fold,0



[DeBERTa] === fold 1 (train=1000, val=1000) ===


Loading weights: 100%|██████████| 390/390 [00:00<00:00, 22255.19it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not 

    step 0/250  loss=0.6235  elapsed=1s
    step 25/250  loss=0.4831  elapsed=17s
    step 50/250  loss=0.4855  elapsed=33s
    step 75/250  loss=0.3839  elapsed=49s
    step 100/250  loss=0.5449  elapsed=65s
    step 125/250  loss=0.1987  elapsed=81s
    step 150/250  loss=0.3455  elapsed=97s
    step 175/250  loss=0.0954  elapsed=113s
    step 200/250  loss=0.0801  elapsed=129s
    step 225/250  loss=0.0317  elapsed=145s
  ep0 (222s)  train_loss=0.2957 val_loss=0.0750 val_map3=0.9498 val_acc=0.9180 val_f1=0.9135
    step 0/250  loss=0.0000  elapsed=1s
    step 25/250  loss=0.2586  elapsed=17s
    step 50/250  loss=0.0000  elapsed=33s
    step 75/250  loss=0.1559  elapsed=49s
    step 100/250  loss=0.0026  elapsed=65s
    step 125/250  loss=0.0186  elapsed=81s
    step 150/250  loss=0.0240  elapsed=97s
    step 175/250  loss=0.0031  elapsed=113s
    step 200/250  loss=0.0000  elapsed=129s
    step 225/250  loss=0.0517  elapsed=145s
  ep1 (221s)  train_loss=0.0304 val_loss=0.0530 val_m

deb/best_val_map3,▁
deb/epoch,▁█
deb/fold,▁▁
deb/train_loss,█▁
deb/val_acc,▁█
deb/val_f1,▁█
deb/val_loss,█▁
deb/val_map3,▁█
deb/best_val_map3,0.95817
deb/epoch,1
deb/fold,1



[DeBERTa] OOF MAP@3=0.9514 acc=0.9235 f1=0.9172
Saved test_deb.npy + oof_deb.npy.


In [7]:
push_model(
    "deberta",
    "/kaggle/working/artifacts/deberta",
    "v3"
)

[hub] uploading 'deberta' -> kamal134134/genai_project/pyTorch/deberta
Uploading Model https://api.kaggle.com/models/kamal134134/genai_project/pyTorch/deberta ...
Starting upload for file /kaggle/working/artifacts/deberta/best_deberta_fold1.pth


Uploading: 100%|██████████| 1.74G/1.74G [00:16<00:00, 108MB/s] 

Upload successful: /kaggle/working/artifacts/deberta/best_deberta_fold1.pth (2GB)
Starting upload for file /kaggle/working/artifacts/deberta/oof_deb.npy



Uploading: 100%|██████████| 40.1k/40.1k [00:00<00:00, 144kB/s]

Upload successful: /kaggle/working/artifacts/deberta/oof_deb.npy (39KB)
Starting upload for file /kaggle/working/artifacts/deberta/best_deberta_fold0.pth



Uploading: 100%|██████████| 1.74G/1.74G [00:16<00:00, 108MB/s] 

Upload successful: /kaggle/working/artifacts/deberta/best_deberta_fold0.pth (2GB)
Starting upload for file /kaggle/working/artifacts/deberta/test_deb.npy



Uploading: 100%|██████████| 10.1k/10.1k [00:00<00:00, 40.4kB/s]

Upload successful: /kaggle/working/artifacts/deberta/test_deb.npy (10KB)


Your model instance version has been created.
Files are being processed...
See at: https://api.kaggle.com/models/kamal134134/genai_project/pyTorch/deberta
Done. Inference notebooks can now kagglehub.
